Run: docker compose up -d

In [1]:
import numpy as np

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from sentence_transformers import SentenceTransformer


In [2]:
METRIC_TYPE = "COSINE" # Która metoda ma być użyta do porównywania
TEXT_TO_SEARCH = 30         # Numer sentencji do porównania
NN_NUMBER = 5               # Ilość sąsiadów do odszukania
COLLECTION_NAME = "sentences_collection"

# Lista metod porównujących
METRIC_LIST = {
    "COSINE": Distance.COSINE,
    "EUCLID": Distance.EUCLID,
    "DOT": Distance.DOT
}

In [3]:
# Dane tekstowe
# sentences = [
#     "Słońce świeci na niebie.", # 0
#     "Dzisiaj jest piękna pogoda.",
#     "Lubię uczyć się programowania.",
#     "Python to popularny język programowania.",
#     "Wieczorem często czytam książki.",
#     "Mój pies lubi biegać po parku.",
#     "Programowanie to świetna zabawa.",
#     "Niebo jest dziś bardzo niebieskie.",
#     "Czytanie rozwija wyobraźnię.",
#     "Chodzę codziennie na spacer z psem.",
#     "Lubię Mazury, morze i góry.", # 10
#     "Java to mój ulubiony język programowania.",
#     "Ostatnio częściej programuję w Pythonie",
#     "SQL też nie jest zły.",
#     "Za tydzień jedziemy całą ekipą na wakacje w góry.",
#     "Zawsze na wyjazdy wakacyjne zabieramy naszego psa.",
#     "Dostałem wczoraj mandat za przektoczenie predkości.",
#     "Jazda moim Porshe wyzwala we mnie mega emocje.",
#     "Kolarstwo górskie daje mi wiele frajdy.",
#     "Wczoraj byłem w kinie.",
#     "Mam uczulenie na czekoladę ale za to mogę jeść marmoladę.", # 20
#     "Najlepsze pączki są z marmoladą i lukrem.",
#     "Twój tort urodzinowy był pyszny.",
#     "Od roku ograniczam cukier i czuję sie wyśmienicie.",
#     "Wczoraj kupiłem poradnik pt. Dieta dla każdego.",
#     "Sposób odżywiania ma wpływ na nasze samopoczucie.",
#     "Właśnie skończyłem czytać fajną książkę.",
#     "Czytanie książek to moja ulubiona zabawa.",
#     "Wczoraj byłem w bibliotek i wypożyczyłem kilka książek na temat programowania.",
#     "Wczoraj wróciliśmy z wakacji z Mazur.",
#     "Mój pies lubi gonić koty." # 30
# ]

sentences = [
    "Wireless noise cancelling headphones with long battery life",
    "Bluetooth over-ear headphones with active noise cancellation",
    "Portable wireless speaker with deep bass and Bluetooth connectivity",
    "Smartphone with high resolution camera and fast processor",
    "Gaming laptop with powerful GPU and high refresh rate display",
    "Lightweight laptop for everyday work and browsing",
    "Mechanical keyboard with RGB lighting for gaming",
    "Ergonomic wireless mouse with adjustable DPI",
    "4K monitor with ultra-thin bezels and HDR support",
    "USB-C docking station for multiple devices",

    "Running shoes designed for long distance comfort",
    "Lightweight sneakers for everyday casual wear",
    "Breathable sports t-shirt for training sessions",
    "Fitness tracker with heart rate monitoring",
    "Smartwatch with activity tracking and notifications",
    "Yoga mat with non-slip surface",
    "Dumbbells set for strength training at home",
    "Resistance bands for full body workout",
    "Cycling helmet with aerodynamic design",
    "Sports water bottle with leak-proof design",

    "Modern coffee machine with programmable settings",
    "Automatic espresso machine with milk frother",
    "Compact vacuum cleaner with strong suction power",
    "Robot vacuum cleaner with smart navigation",
    "Air purifier with HEPA filter for home use",
    "Table lamp with adjustable brightness levels",
    "LED desk lamp with touch control",
    "Smart thermostat for energy saving",
    "Electric kettle with temperature control",
    "Blender for smoothies and healthy drinks",

    "Casual denim jeans with comfortable fit",
    "Stylish leather jacket for autumn season",
    "Winter jacket with thermal insulation",
    "Classic white t-shirt made from cotton",
    "Sneakers with modern design and comfort",
    "Elegant dress for special occasions",
    "Hoodie with soft fabric and relaxed fit",
    "Running shorts for summer workouts",
    "Formal shirt for business meetings",
    "Sports jacket with breathable material",

    "Bestselling novel with engaging storyline",
    "Science fiction book with futuristic themes",
    "Cookbook with healthy recipes and tips",
    "Biography of a famous entrepreneur",
    "Self-help book about productivity and habits",
    "Guide to learning programming for beginners",
    "Fantasy novel with magical world",
    "Thriller book with unexpected twists",
    "Educational book for children",
    "Travel guide for exploring Europe",

    "Educational toy for kids with interactive features",
    "Building blocks set for creative play",
    "Puzzle game for brain training",
    "Board game for family entertainment",
    "Remote control car with high speed",
    "Interactive robot toy with sensors",
    "Plush toy for children",
    "STEM learning kit for kids",
    "Toy train set with tracks",
    "Outdoor play set for children",

    "Noise cancelling earbuds with compact design",
    "Wireless earbuds with long battery life",
    "Portable charger with fast charging support",
    "Smartphone case with shock protection",
    "Tablet device for media consumption",
    "Gaming console with high performance graphics",
    "VR headset for immersive gaming experience",
    "Smart home hub with voice control",
    "Wireless router with high speed internet",
    "External SSD with fast data transfer",

    "Trail running shoes for outdoor adventures",
    "Hiking backpack with large capacity",
    "Camping tent for 2 people",
    "Sleeping bag for cold weather",
    "Portable gas stove for camping",
    "Climbing shoes with strong grip",
    "Fitness gloves for weight lifting",
    "Jump rope for cardio training",
    "Exercise bike for home workouts",
    "Treadmill with adjustable speed",

    "Smart LED light bulbs with app control",
    "Home security camera with motion detection",
    "Door lock with fingerprint access",
    "Video doorbell with HD camera",
    "Smart plug with remote control",
    "Air fryer for healthy cooking",
    "Microwave oven with multiple functions",
    "Dishwasher with energy saving mode",
    "Refrigerator with large storage capacity",
    "Washing machine with quick wash program"
]

In [6]:
# Tworzenie embeddingów
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(sentences).astype(np.float32)

# Inicjalizacja klienta Qdrant (lokalnie)
client = QdrantClient(host="qdrant_nosql_lab", port=6333)

# Nazwa kolekcji
collection_name = COLLECTION_NAME

# Tworzenie kolekcji z metryką cosine
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=embeddings.shape[1], distance=METRIC_LIST[METRIC_TYPE])
    )

# Dodanie punktów (embedding + tekst jako payload)
points = [
    PointStruct(id=i, vector=embeddings[i], payload={"text": sentences[i]})
    for i in range(len(sentences))
]

client.upsert(collection_name=collection_name, points=points)

# Zapytanie


# query_vector = embeddings[TEXT_TO_SEARCH]

# print(f"Zapytanie: {sentences[TEXT_TO_SEARCH]}")

text = 'Hoodie with hard fabric and modern fit'
query_vector = model.encode([text]).astype(np.float32)[0]
print(f'zapytanie: {text}, vector: {query_vector}')


# Wyszukiwanie 3 najbliższych
results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=NN_NUMBER,
    with_payload=True
)

# Wyniki
print("Najbardziej podobne zdania:")
for hit in results:
    for item in hit[1]:
        # print(item)
        text = item.payload["text"]
        score = item.score
        print(f"- \"{text}\" (similarity: {score:.4f})")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


zapytanie: Hoodie with hard fabric and modern fit, vector: [-8.09396878e-02  7.10356757e-02 -8.81722383e-03  5.99914268e-02
  7.49946684e-02 -1.43130254e-02  8.52987170e-02  2.97593959e-02
 -7.37035573e-02  4.36852276e-02 -6.75061392e-03 -3.69577967e-02
  1.02774873e-02  5.89423534e-03  8.57923273e-03 -6.12583123e-02
  2.46496201e-02 -2.19154675e-02 -2.32432149e-02 -2.48022377e-02
 -1.39771983e-01 -3.65409292e-02  1.55280046e-02  1.02005657e-02
 -2.39723120e-02 -2.96983719e-02 -1.14018396e-02  2.40367465e-02
  6.17614528e-03 -3.96883674e-02  1.00198924e-03  8.13473016e-03
  4.82761447e-04  3.83210368e-02 -3.92089821e-02 -2.07074508e-02
  7.19413683e-02  2.28984077e-02 -2.71040089e-02  8.42052395e-04
 -9.51680075e-03 -2.77550779e-02 -5.68606704e-02 -4.32877354e-02
  6.50899038e-02 -2.76686940e-02  1.41266063e-02  1.41986355e-01
  7.37985522e-02 -2.07692827e-03  1.25908069e-02 -6.58490360e-02
  1.59421097e-02 -2.61849351e-03 -3.39619368e-02  2.57492252e-02
  2.12575626e-02  3.78853232e-0

In [7]:
client.delete_collection(collection_name=COLLECTION_NAME)

True

In [8]:
collections = client.get_collections()
print(collections)

collections=[CollectionDescription(name='my_collection2')]
